In [ ]:
'''   
K Nearest Neighbors
-------------------------------
-Both Classification and Regression
Clssification - KNeighbourClassifier
Regression - KNeighbourRegressor

- Classification
  - calculates distance between data points to choose nearest neighbors to identitfy new data belongs to which class

- Regression
   - calculates distance between data points to get average data as a prediction value

Finding optimal number of neighbours - Using Elbow Method Figure
---------------------------------------------------------
1. Error rate
   - maximum drop point of figure.

2. Accuracy Score 
    - Highest Peak point of figure

Calculation of Elbow Method
---------------------------------------------
-Error Rate
  -> Number of wrong predictions

- Accuracy Score
  -> Accurcy of each execution of model   


Classification
----------------------------------
Suppose
   k_range = range(1,21)  # 1-20
   error_rate =[]
   acc_score= []

   for i in k_range:
    knn= KNeighborClassifier()
    knn.fit(X_train_scale, Y_train)
    knn_pred= knn-predict(X_test_scale)


    error_rate.append(np.mean(Y_test != knn_pred)) -> number of wrong prediction/ tottal number of Y_test data
    acc_score.append(Y_test, knn_pred)




    
Cross Validation
-------------------------------
- StratifyFold
- cross_val_score


Overfitting and Underfitting
---------------------------------------
overfit -> too good in traning data but performs bad in testing
undefit -> performs poorly on both traing and tesing data

'''

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# import model
from sklearn.neighbors import KNeighborsClassifier

# for splitting
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV

# for scaling of data
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [57]:
df = pd.read_csv('Dataset/student.csv')
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,higher education,higher education,at_home,teacher,...,very good,medium,high,very low,very low,good,6,5,6,6
1,GP,F,17,U,GT3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,excellent,medium,medium,very low,very low,good,4,5,5,6
2,GP,F,15,U,LE3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,very good,medium,low,low,medium,good,10,7,8,10
3,GP,F,15,U,GT3,T,higher education,5th to 9th grade,health,services,...,good,low,low,very low,very low,excellent,2,15,14,15
4,GP,F,16,U,GT3,T,secondary education,secondary education,other,other,...,very good,medium,low,very low,low,excellent,4,6,10,10


In [58]:
# Changing grade to Label
df['G3'].sort_values(ascending= False).unique()



array([20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10,  9,  8,  7,  6,  5,  4,
        0])

In [ ]:
def convert_grade(grade):
    if grade >= 15:
        return 'A'
    elif grade >= 11:
        return 'B'
    elif grade>= 7:
        return 'C'

    else:
        return 'D'

In [60]:
df['Grade']= df['G3'].apply(convert_grade)

In [61]:
df['Grade'].value_counts()

Grade
B    169
C    140
D     46
A     40
Name: count, dtype: int64

In [62]:
df.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3', 'Grade'],
      dtype='str')

In [63]:
columns= df[['Medu', 'Fedu', 'traveltime', 'studytime', 'famsup', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2','Grade']]
columns_to_encode=df[['failures', 'schoolsup','internet','famrel']]

Label ENcoding

In [64]:
'''  
1. fit_transform() -> convert values to 0,1,2,3, ---n
2. inverse_trandform() -> convert converted number back to label data.
    data = encoder.inverse_tranform(df['failures_id])
    data

'''

encoder = LabelEncoder()


for col in columns_to_encode:
    df[f'{col}_id']= encoder.fit_transform(df[col])
print('Data COnverted')



Data COnverted


In [138]:
df

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,Medu_id,Fedu_id,traveltime_id,studytime_id,famsup_id,freetime_id,goout_id,Dalc_id,health_id,Grade_id
0,GP,F,18,U,GT3,A,higher education,higher education,at_home,teacher,...,4.0,4.0,1.0,1,0,2.0,3.0,0.0,2.0,2
1,GP,F,17,U,GT3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,4.0,3.0,0.0,1,1,2.0,2.0,0.0,2.0,2
2,GP,F,15,U,LE3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,4.0,3.0,0.0,1,0,2.0,1.0,1.0,2.0,2
3,GP,F,15,U,GT3,T,higher education,5th to 9th grade,health,services,...,4.0,3.0,0.0,2,1,1.0,1.0,0.0,2.0,1
4,GP,F,16,U,GT3,T,secondary education,secondary education,other,other,...,2.0,2.0,0.0,1,1,2.0,1.0,0.0,2.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,MS,M,20,U,LE3,A,5th to 9th grade,5th to 9th grade,services,services,...,3.0,3.0,0.0,1,1,2.0,3.0,3.0,3.0,2
391,MS,M,17,U,LE3,T,secondary education,primary education (4th grade),services,services,...,2.0,3.0,1.0,0,0,3.0,2.0,2.0,1.0,0
392,MS,M,21,R,GT3,T,primary education (4th grade),primary education (4th grade),other,other,...,4.0,3.0,0.0,0,0,2.0,2.0,2.0,2.0,2
393,MS,M,18,R,LE3,T,secondary education,5th to 9th grade,services,other,...,2.0,3.0,0.0,0,0,3.0,0.0,2.0,2.0,2


Mannual Encode

In [66]:
def encodeData( dataframe,col_name, *args):
    encode_data= {}
    for idx,val in enumerate(args):
        encode_data[val]= idx

    dataframe[f'{col_name}_id']= dataframe[col_name].map(encode_data)
    print(f'{col_name} data encoded')

In [67]:
df['Medu'].unique()

<StringArray>
[             'higher education', 'primary education (4th grade)',
           'secondary education',              '5th to 9th grade',
                          'none']
Length: 5, dtype: str

In [68]:
encodeData(df, 'Medu','none','primary education(4th grade)','secondary education','5th to 9th grade','higher education')

Medu data encoded


In [69]:
df['Fedu'].unique()

<StringArray>
[             'higher education', 'primary education (4th grade)',
              '5th to 9th grade',           'secondary education',
                          'none']
Length: 5, dtype: str

In [70]:
encodeData(df, 'Fedu','none','primary education(4th grade)','secondary education','5th to 9th grade','higher education')

Fedu data encoded


In [71]:
df['traveltime'].unique()

<StringArray>
['15 to 30 min.', '<15 min.', '30 min. to 1 hour', '>1 hour']
Length: 4, dtype: str

In [ ]:
encodeData(df, 'traveltime','<15 min.','15 to 30 min.','30 min. to 1 hour','>1 hour')

traveltime data encoded


In [73]:
df['studytime'].unique()

<StringArray>
['2 to 5 hours', '5 to 10 hours', '<2 hours', '>10 hours']
Length: 4, dtype: str

In [74]:
encodeData(df, 'studytime','<2 hours','2 to 5 hours','5 to 10 hours','>10 hours')

studytime data encoded


In [75]:
df['famsup'].unique()

<StringArray>
['no', 'yes']
Length: 2, dtype: str

In [76]:
encodeData(df, 'famsup','no','yes')

famsup data encoded


In [77]:
df['freetime'].unique()

<StringArray>
['medium', 'low', 'high', 'very low', 'very high']
Length: 5, dtype: str

In [78]:
encodeData(df, 'freetime','very low','low','medium','high',' very high')

freetime data encoded


In [79]:
df['goout'].unique()

<StringArray>
['high', 'medium', 'low', 'very low', 'very high']
Length: 5, dtype: str

In [80]:
encodeData(df, 'goout','very low','low','medium','high',' very high')

goout data encoded


In [81]:
df['Dalc'].unique()

<StringArray>
['very low', 'low', 'very high', 'medium', 'high']
Length: 5, dtype: str

In [82]:
encodeData(df, 'Dalc','very low','low','medium','high',' very high')

Dalc data encoded


In [83]:
df['Walc'].unique()

<StringArray>
['very low', 'medium', 'low', 'high', 'very high']
Length: 5, dtype: str

In [84]:
encodeData(df, 'Dalc','very low','low','medium','high',' very high')

Dalc data encoded


In [85]:
df['health'].unique()

<StringArray>
['good', 'excellent', 'very bad', 'bad', 'very good']
Length: 5, dtype: str

In [ ]:
encodeData(df, 'health','very bad','bad','good','very good','excellent')

health data encoded


In [87]:
df['absences'].unique()

array([ 6,  4, 10,  2,  0, 16, 14,  7,  8, 25, 12, 54, 18, 26, 20, 56, 24,
       28,  5, 13, 15, 22,  3, 21,  1, 75, 30, 19,  9, 11, 38, 40, 23, 17])

In [88]:
df['G1'].unique()

array([ 5,  7, 15,  6, 12, 16, 14, 10, 13,  8, 11,  9, 17, 19, 18,  4,  3])

In [89]:
df['G2'].unique()

array([ 6,  5,  8, 14, 10, 15, 12, 18, 16, 13,  9, 11,  7, 19, 17,  4,  0])

In [90]:
df['Grade'].unique()

<StringArray>
['C', 'B', 'A', 'D']
Length: 4, dtype: str

In [ ]:
encodeData(df, 'Grade','D','C','B','A')

Grade data encoded


In [92]:
df

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,Medu_id,Fedu_id,traveltime_id,studytime_id,famsup_id,freetime_id,goout_id,Dalc_id,health_id,Grade_id
0,GP,F,18,U,GT3,A,higher education,higher education,at_home,teacher,...,4.0,4.0,1.0,1,0,2.0,3.0,0.0,2.0,2
1,GP,F,17,U,GT3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,NaN,NaN,0.0,1,1,2.0,2.0,0.0,2.0,2
2,GP,F,15,U,LE3,T,primary education (4th grade),primary education (4th grade),at_home,other,...,NaN,NaN,0.0,1,0,2.0,1.0,1.0,2.0,2
3,GP,F,15,U,GT3,T,higher education,5th to 9th grade,health,services,...,4.0,3.0,0.0,2,1,1.0,1.0,0.0,NaN,1
4,GP,F,16,U,GT3,T,secondary education,secondary education,other,other,...,2.0,2.0,0.0,1,1,2.0,1.0,0.0,NaN,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,MS,M,20,U,LE3,A,5th to 9th grade,5th to 9th grade,services,services,...,3.0,3.0,0.0,1,1,NaN,3.0,3.0,3.0,2
391,MS,M,17,U,LE3,T,secondary education,primary education (4th grade),services,services,...,2.0,NaN,1.0,0,0,3.0,NaN,2.0,1.0,0
392,MS,M,21,R,GT3,T,primary education (4th grade),primary education (4th grade),other,other,...,NaN,NaN,0.0,0,0,NaN,2.0,2.0,2.0,2
393,MS,M,18,R,LE3,T,secondary education,5th to 9th grade,services,other,...,2.0,3.0,NaN,0,0,3.0,0.0,2.0,NaN,2


In [93]:
df.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3', 'Grade', 'failures_id',
       'schoolsup_id', 'internet_id', 'famrel_id', 'Medu_id', 'Fedu_id',
       'traveltime_id', 'studytime_id', 'famsup_id', 'freetime_id', 'goout_id',
       'Dalc_id', 'health_id', 'Grade_id'],
      dtype='str')

#### Feature and Target

In [119]:
feature= ['Medu_id', 'Fedu_id',
       'traveltime_id', 'studytime_id','famsup_id', 'freetime_id','schoolsup_id', 'internet_id','goout_id',
       'Dalc_id', 'health_id','absences', 'G1', 'G2']
target= 'Grade_id'

In [120]:
X= df[feature]
Y= df[target]

In [121]:
X

,Medu_id,Fedu_id,traveltime_id,studytime_id,famsup_id,freetime_id,schoolsup_id,internet_id,goout_id,Dalc_id,health_id,absences,G1,G2
0,4.0,4.0,1.0,1,0,2.0,1,0,3.0,0.0,2.0,6,5,6
1,NaN,NaN,0.0,1,1,2.0,0,1,2.0,0.0,2.0,4,5,5
2,NaN,NaN,0.0,1,0,2.0,1,1,1.0,1.0,2.0,10,7,8
3,4.0,3.0,0.0,2,1,1.0,0,1,1.0,0.0,NaN,2,15,14
4,2.0,2.0,0.0,1,1,2.0,0,0,1.0,0.0,NaN,4,6,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,3.0,3.0,0.0,1,1,NaN,0,0,3.0,3.0,3.0,11,9,9
391,2.0,NaN,1.0,0,0,3.0,0,1,NaN,2.0,1.0,3,14,16
392,NaN,NaN,0.0,0,0,NaN,0,0,2.0,2.0,2.0,3,10,8
393,2.0,3.0,NaN,0,0,3.0,0,1,0.0,2.0,NaN,0,11,12


In [122]:
X.isnull().sum()

Medu_id           59
Fedu_id           82
traveltime_id     23
studytime_id       0
famsup_id          0
freetime_id       40
schoolsup_id       0
internet_id        0
goout_id          53
Dalc_id            9
health_id        146
absences           0
G1                 0
G2                 0
dtype: int64

In [123]:
df['Medu_id'] = df['Medu_id'].replace(['null', 'NaN', 'None', '?'], np.nan)
medu_mode=df['Medu_id'].mode()[0]
medu_mode

np.float64(4.0)

In [124]:
fedu_mode=df['Fedu_id'].mode()[0]
fedu_mode

np.float64(3.0)

In [125]:
traveltime_mode=df['traveltime_id'].mode()[0]
traveltime_mode

np.float64(0.0)

In [126]:
freetime_mode=df['freetime_id'].mode()[0]
freetime_mode

np.float64(2.0)

In [127]:
goout_mode=df['goout_id'].mode()[0]
goout_mode

np.float64(2.0)

In [128]:
Dalc_mode=df['Dalc_id'].mode()[0]
Dalc_mode

np.float64(0.0)

In [129]:
health_mode=df['health_id'].mode()[0]
health_mode

np.float64(2.0)

In [130]:
X['Medu_id'].unique()

array([ 4., nan,  2.,  3.,  0.])

In [133]:
df['Fedu_id']=df['Fedu_id'].fillna(fedu_mode)
df['Medu_id']=df['Medu_id'].fillna(medu_mode)
df['traveltime_id']=df['traveltime_id'].fillna(traveltime_mode)
df['freetime_id']=df['freetime_id'].fillna(freetime_mode)
df['goout_id']=df['goout_id'].fillna(goout_mode)
df['Dalc_id']=df['Dalc_id'].fillna(Dalc_mode)
df['health_id']=df['health_id'].fillna(health_mode)

In [134]:
X.isnull().sum()

Medu_id           59
Fedu_id           82
traveltime_id     23
studytime_id       0
famsup_id          0
freetime_id       40
schoolsup_id       0
internet_id        0
goout_id          53
Dalc_id            9
health_id        146
absences           0
G1                 0
G2                 0
dtype: int64

### Train and Test

In [135]:
X_train, X_test, Y_train, Y_test= train_test_split(
    X,Y,test_size=0.2, random_state= 42, stratify= Y
)

### feature Scaling

In [136]:
scaler= StandardScaler()
X_train_scale= scaler.fit_transform(X_train)
X_test_scale= scaler.transform(X_test)

### Elbow Method

In [137]:
k_range= range(1,21)
error_rate= []
acc_score=[]

for i in k_range:
    knn= KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train_scale, Y_train)
    knn_pred= knn.predict(X_test_scale)


    error_rate.append(np.mean(Y_test != knn_pred))
    acc_score.append(accuracy_score(Y_test, knn_pred))

ValueError: Input X contains NaN.
KNeighborsClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

### Cross Validation

In [ ]:
kfold= StratifiedKFold()

### GridSearch CV 

In [ ]:
'''   
- uniform -> reduces baisness by taking mean value
- euclidean -> for continuous data + dense data
- manhattan -> for ordinal data
- euclidean + manhattan -> for nominal data 

'''


param= {
    'n_neighbors':list(range(1,21)),
    'weights': ['uniform','distance'],
    'metric':['euclidean','manhattan']
}

In [ ]:
grid_cv= GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid= param,
    cv= kfold,
    scoring= 'accuracy',
    n_jobs=1   #use 1 core of cpu
)

In [ ]:
grid_cv.fit(X_train_scale, Y_train)

In [ ]:
best_knn= grid_cv.best_estimator
best_knn_pred= best_knn.predict(X_test_scale)

In [ ]:
best_cr= classification_report(Y_test, best_knn_pred)
best_cm= confusion_matrix(Y_test, best_knn_pred)

In [ ]:
print(best_cm)
print(best_cr)